# Module 18 — pandas

Fifty readings in `data/readings.csv`, and questions to ask of them. Module 08 read a
file like this a line at a time; this module reads it as a **table** and works on
whole columns at once.

Two things to watch for.

**pandas guesses the type of every column from the data**, and one unreadable cell in
fifty changes the type of the whole column. What happens next is the fourth appearance
of this course's recurring failure — except that this time it is not consistent, and
the inconsistency is the lesson.

**This is also the first module where a notebook earns its keep**, and section 8 says
why it did not earlier.

In [ ]:
from pathlib import Path

import pandas as pd

HERE = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd() / "18_pandas"
READINGS = HERE / "data" / "readings.csv"

print(pd.__version__)
print(READINGS.exists())

## 1. A DataFrame is a table with typed columns

`read_csv` does in one line what module 08 did in five — and makes a decision module
08 never made: it picks a **type per column**.

In [ ]:
frame = pd.read_csv(READINGS, sep=";")

print(frame.shape)  # (rows, columns)
print(frame.dtypes.to_string())

`frame.shape` is `(50, 5)`. And look at `value`: **`str`**, not a number. Module 08's
CSV gave you strings too, but there you converted each one yourself and a bad field
raised at the line that touched it. Here the whole column is text because *one* cell in
it is, and nothing said so.

`frame.head()` and `frame.tail()` show the ends; `frame.info()` is the one to run
first on a file you have not seen.

In [ ]:
frame = pd.read_csv(READINGS, sep=";")

print(frame.head(3).to_string())
print()
frame.info()

## 2. One bad cell, and what pandas does about it

The file has three unreadable values in fifty. Two of them are `n/a`, which pandas
recognises; one is `kaputt`, which it does not.

Predict what arithmetic on the `value` column gives.

In [ ]:
frame = pd.read_csv(READINGS, sep=";")

# The column is text. What does .sum() do to a column of text?
assert type(frame["value"].sum()).__name__ == ...

In [ ]:
frame = pd.read_csv(READINGS, sep=";")

print("sum():", repr(frame["value"].sum())[:60], "...")
print("max():", repr(frame["value"].max()))

for operation in ("mean", "median"):
    try:
        getattr(frame["value"], operation)()
    except TypeError as err:
        print(f"{operation}(): TypeError: {err}")

try:
    frame["value"] > 85
except TypeError as err:
    print("> 85:", type(err).__name__, "-", err)

print('> "85":', (frame["value"] > "85").sum(), "rows -- compared as text, not as numbers")

Read that carefully, because it is not one behaviour:

| | |
| --- | --- |
| `.sum()` | **concatenated the strings.** No error, and the result starts `20.722.4…` |
| `.max()` | gave `'n/a'` — the largest *string*, alphabetically |
| `.mean()` | raised `TypeError` |
| `> 85` | raised `TypeError` |
| `> "85"` | compared **as text**, and said 5 rows. No error |

So pandas raises for some of these and quietly does something else for others, and
there is no rule you can hold in your head about which. `.sum()` on a text column is
the dangerous one: it returns a value, of a plausible type, that is not a total of
anything.

`.describe()` is the fastest way to notice, because it changes shape entirely:

In [ ]:
frame = pd.read_csv(READINGS, sep=";")

print("text column:")
print(frame["value"].describe().to_string())

`count`, `unique`, `top`, `freq` — this is what `describe()` gives for a text column.
If you expected `mean` and `std` and got these four, the column is not what you think
it is. **That is the check to run**, and it costs one line.

Two ways to fix it. At read time, by naming what counts as missing:

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

print(frame.dtypes.to_string())
print()
print(frame["value"].describe().round(2).to_string())

Or afterwards, on a column you have already got:

In [ ]:
frame = pd.read_csv(READINGS, sep=";")

frame["value"] = pd.to_numeric(frame["value"], errors="coerce")

print(frame["value"].dtype)
print(frame["value"].isna().sum(), "of", len(frame), "could not be read")
print("without errors='coerce' it would raise -- which is often what you want")

`errors="coerce"` turns what cannot be parsed into `NaN`; the default raises. Which
you want depends on whether a bad cell is data or a bug — and that is a decision, so
make it deliberately rather than by leaving the argument out.

`na_values` is worth knowing precisely. pandas already treats these as missing:
`""`, `NA`, `N/A`, `n/a`, `NaN`, `nan`, `null`, `NULL`, `None`, `<NA>` and a few
Excel spellings. It does **not** treat `-`, `?`, `unknown`, `missing` or `kaputt` as
missing, and every dataset has its own convention.

## 3. Selecting

Two objects: a **`DataFrame`** is the table, a **`Series`** is one column. A column by
name gives a Series; a list of names gives a DataFrame.

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

print(type(frame["value"]).__name__)
print(type(frame[["tag", "value"]]).__name__)
print(frame["tag"].unique())
print(frame["location"].value_counts().to_string())

Rows come out by **condition**, not by index. The condition is itself a Series — of
booleans, one per row — and that is worth seeing before it becomes invisible:

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

hot = frame["value"] > 85  # a Series of True/False, one per row
print(type(hot).__name__, hot.dtype, hot.sum(), "rows are True")

print(frame.loc[hot, ["tag", "value", "at"]].to_string(index=False))

`.loc[rows, columns]` takes the mask and the column names. Two rules that save time:

- **Combine conditions with `&` and `|`, and bracket them.** `and` and `or` do not
  work on a Series — they want a single truth value, and a Series of fifty has none.
  Module 02's `and`/`or` returning an operand is exactly why: there is no operand to
  return.
- **`.loc` is by label, `.iloc` is by position.** For a default index they look the
  same, which is how people learn the wrong one.

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

try:
    frame[(frame["value"] > 85) and (frame["location"] == "Test rig")]
    outcome = "worked"
except ValueError as err:
    outcome = type(err).__name__

# What does `and` between two Series do?
assert outcome == ...

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

both = frame[(frame["value"] > 85) & (frame["location"] == "Test rig")]
print(len(both), "rows")
print(both[["tag", "value"]].to_string(index=False))

## 4. `NaN`, and what it does to an answer

A missing value is `NaN`, and it is not zero and not `None`. Three behaviours worth
knowing, because they are not the same as each other.

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

print("rows:          ", len(frame))
print("value.count(): ", frame["value"].count(), "-- count() skips NaN")
print("value.isna():  ", frame["value"].isna().sum())
print("mean over 47:  ", round(frame["value"].mean(), 3), "-- mean() skips NaN too")
print("sum of a mask: ", (frame["value"] > 85).sum(), "-- a NaN is not > 85, and not counted")
print("NaN == NaN:    ", float("nan") == float("nan"), "-- so never test with ==")
print("pd.isna(NaN):  ", pd.isna(float("nan")))

So `mean()` gives the mean **of what is there**, which is usually what you want and is
never what you should assume. `len(frame)` is 50 and `count()` is 47: if a report says
"average of 50 readings" and the code called `.mean()`, the report is wrong by three.

`dropna()` and `fillna()` are the two decisions available, and they are decisions:

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

print(len(frame["value"].dropna()), "left after dropna()")
print(
    round(frame["value"].fillna(0).mean(), 3), "-- mean after fillna(0): three zeroes pull it down"
)
print(round(frame["value"].mean(), 3), "-- mean with NaN skipped")

`fillna(0)` on a temperature is almost always wrong: it invents three readings of
freezing point and moves the mean. `dropna()` says "I have 47 readings", which is
true. The general rule: **filling a missing value is a claim about the world**, so it
needs a reason you could defend.

## 5. Grouping

The operation that makes a table worth having: split by a column, compute per group,
put it back together.

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

print(frame.groupby("location")["value"].mean().round(2).to_string())

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

summary = frame.groupby("location")["value"].agg(["count", "mean", "max"]).round(2)
print(summary.to_string())
print()
print(summary.index.name, "->", summary.index.tolist())

`count` per group is worth having in every summary for the reason section 4 gave: a
mean over nine readings and a mean over twenty are not comparable, and the count is
what tells you which you have.

`groupby` on more than one column gives a row per combination, and `sort_values` puts
the interesting end first:

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

per_tag = frame.groupby(["location", "tag"])["value"].agg(["count", "max"]).round(1)
print(per_tag.sort_values("max", ascending=False).head(5).to_string())

## 6. Time

`parse_dates=` turned the `at` column into timestamps rather than text, which is what
makes the next line possible: `.dt` reaches the parts of a datetime the way `.str`
reaches the parts of a string.

In [ ]:
frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])

print(frame["at"].dtype)
print(frame["at"].min(), "to", frame["at"].max())

frame["hour"] = frame["at"].dt.hour
print(frame.groupby("hour")["value"].mean().round(1).to_string())

Without `parse_dates` that column would be text, `.dt` would raise, and `min()` would
give you the alphabetically smallest string — which for ISO-8601 timestamps happens to
be the earliest, so it would look right and be right by luck.

## 7. Writing the answer back out

`to_csv` and `to_json` are `read_csv` backwards, and `index=False` is the argument to
remember: without it the row numbers become a column in the file, and whoever reads it
next gets an extra field called `Unnamed: 0`.

In [ ]:
import tempfile

frame = pd.read_csv(READINGS, sep=";", na_values=["kaputt"], parse_dates=["at"])
summary = frame.groupby("location")["value"].agg(["count", "mean", "max"]).round(2)

target = Path(tempfile.mkdtemp()) / "summary.csv"
summary.to_csv(target)  # the index is the location, so here it belongs in the file

print(target.read_text(encoding="utf-8"))
print(pd.read_csv(target).columns.tolist())

## 8. Why the notebook, and why only now

Seventeen modules of `.py` files, and now a module where the notebook is the point.
The reason is not that pandas is graphical. It is the shape of the work:

**A notebook pays for itself when the state is expensive and the questions are
cheap.** Reading a two-million-row CSV takes forty seconds; asking it fifty questions
takes a fifth of a second each. In a script you pay the forty seconds again for every
question. In a notebook you pay once and then ask.

That is why it would have been a bad idea in module 01, where the state was
`name = "Max"` and cost nothing. Where the state is free, the kernel is pure overhead
and the out-of-order execution is pure risk.

Which brings the risk into the open, because it is real:

- **Cells can run in any order**, and the numbers beside them tell you the order they
  *were* run in, not the order they appear. A notebook that works can stop working
  when it is run top to bottom, which is the only way anybody else will run it.
- **State outlives the code.** Delete the cell that defined `frame` and `frame` is
  still there. The notebook now depends on a line that no longer exists.

The discipline that fixes both: **Restart & Run All before you believe it**, and again
before you show it to anybody. If it does not survive that, it does not work — it
merely worked once, in an order nobody wrote down.

And the choice, for a given job:

| the job | which | why |
| --- | --- | --- |
| looking for outliers, not sure what you are looking for yet | **notebook** | many cheap questions against one expensive state |
| the same analysis every night at 3 a.m. | **script** | nobody presses Shift+Enter at three in the morning |
| a colleague must reproduce your number in six months | **script** | in a notebook the execution order is not recorded |
| a figure and a paragraph for a report | **notebook** | the output belongs beside the code that made it |

`exercises/thinking.md` asks you to make that call on three cases and defend it.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md`.

Note that the exercises are still `.py` files, checked by an expected output. That is
deliberate: this module is about pandas, and a test that compares text is a better
feedback channel than a notebook. The notebook is where you *explore*, which is what
section 8 just said it is for.

Module 19 puts the same readings in SQLite and asks the same questions in SQL — where
`groupby` is called `GROUP BY` and the answer to "which is better" is genuinely "it
depends".